# ICA 2026 Hackathon — App Sequence Analysis
## Notebook 01: Exploratory Analysis

Walks through loading social media app-usage event data,
building per-user sequences, and running a first pass of sequence analysis.

## 1. Install Dependencies

In [ ]:
%pip install -q numpy pandas scipy networkx scikit-learn statsmodels matplotlib seaborn plotly tqdm python-dotenv

## 2. Imports & Environment Configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src/ to path so we can import project modules
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# Load .env if present
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
    print("Loaded .env")
except ImportError:
    print("python-dotenv not installed — skipping .env load")

DATA_RAW       = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
RESULTS        = REPO_ROOT / "results"

print("REPO_ROOT :", REPO_ROOT)
print("DATA_RAW  :", DATA_RAW)
print("Python    :", sys.version)

## 3. Set Up Data Directories

In [ ]:
for d in [DATA_RAW, DATA_PROCESSED,
          RESULTS / "figures", RESULTS / "tables"]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"OK  {d}")

## 4. Load & Inspect Data

Place your raw data file in `data/raw/`.
If none is present, a synthetic sample is generated so the rest of the notebook is runnable.

In [ ]:
import random
from datetime import datetime, timedelta

raw_files = list(DATA_RAW.glob("*.csv")) + list(DATA_RAW.glob("*.parquet"))

if raw_files:
    path = raw_files[0]
    print(f"Loading {path.name}")
    df_raw = pd.read_csv(path) if path.suffix == ".csv" else pd.read_parquet(path)
else:
    print("No data file found — generating synthetic sample")
    rng  = random.Random(42)
    apps = ["Instagram", "TikTok", "Twitter/X", "Facebook", "YouTube",
            "WhatsApp", "Snapchat", "Reddit", "LinkedIn", "Pinterest"]
    t0   = datetime(2025, 1, 1)
    rows = []
    for _ in range(1000):
        uid = f"u{rng.randint(1, 50):03d}"
        app = rng.choice(apps)
        ts  = t0 + timedelta(seconds=rng.randint(0, 30 * 24 * 3600))
        rows.append({"user_id": uid, "app": app, "timestamp": ts})
    df_raw = pd.DataFrame(rows)

df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"])
print(df_raw.shape)
df_raw.head()

In [ ]:
print(df_raw.dtypes)
print("\nUnique users  :", df_raw["user_id"].nunique())
print("Unique apps   :", df_raw["app"].nunique())
print("Date range    :", df_raw["timestamp"].min(), "—", df_raw["timestamp"].max())
df_raw["app"].value_counts().plot(kind="bar", title="Event counts by app", figsize=(10, 4))
plt.tight_layout()
plt.show()

## 5. Define Sequence Parsing Utilities & Build Sequences

In [ ]:
from data.preprocessing import build_sequences

df_seqs = build_sequences(
    df_raw,
    user_col="user_id",
    app_col="app",
    time_col="timestamp",
)
print(f"{len(df_seqs)} user sequences built")
print("Sequence length stats:")
df_seqs["sequence"].apply(len).describe()

## 6. Top N-grams & Transition Matrix

In [ ]:
from analysis.sequences import top_ngrams

bigrams = top_ngrams(df_seqs["sequence"], n=2, k=15)
print("Top-15 bigrams:")
bigrams

In [ ]:
from visualization.plots import plot_transition_heatmap

fig = plot_transition_heatmap(df_seqs["sequence"].tolist())
fig.savefig(RESULTS / "figures" / "transition_heatmap.png", dpi=150)
plt.show()

## 7. Validate Notebook & Repo Readiness

In [ ]:
checks = {
    "data/raw exists"       : DATA_RAW.exists(),
    "data/processed exists" : DATA_PROCESSED.exists(),
    "results/figures exists": (RESULTS / "figures").exists(),
    "results/tables exists" : (RESULTS / "tables").exists(),
    "df_raw loaded"         : len(df_raw) > 0,
    "sequences built"       : len(df_seqs) > 0,
}
all_ok = True
for check, passed in checks.items():
    icon = "PASS" if passed else "FAIL"
    print(f"[{icon}] {check}")
    all_ok = all_ok and passed

print()
print("Repo ready!" if all_ok else "Some checks failed — see above.")